# Modelling: LightGBM and three other algorithms on the test shop

This notebook trains forecasting models on the feature frame built by [preprocessing.ipynb](preprocessing.ipynb), compares them in a **rolling-window backtest** with the competition metric, and writes a submission. The strategy follows [DATASETS.md](DATASETS.md) §8.

**Before running:** run `preprocessing.ipynb` once. Its last cell saves `processed/features_wide.parquet`, which this notebook loads.

**Models**

| Model | Family | Trained on | Loss / rule | Why it is here |
|---|---|---|---|---|
| `median_13w` | reference | each article's own history | median of the last 13 weeks | the baseline to beat (DATASETS §8.1) |
| `lightgbm` | gradient-boosted trees, **global** | all series (test shop + A + C + D) | L1 on the scaled target | the main model (§8.4). L1 = absolute error, which is what MASE measures |
| `linear_l1` | linear, **global** | same | L1 (least absolute deviation, SGD) | shows how much the trees' non-linearity adds |
| `mlp` | neural network, **global** | same | squared error on `log1p(scaled target)` | a different learner for the ensemble |
| `tsb` | Croston family, **local** | each article's own history | TSB smoothing, median rule | the intermittent-demand specialist (§8.6) |

All global models predict `target / scale`, where `scale` is the series' mean absolute week-to-week change up to the origin. The loss is then close to MASE itself, and a 494-unit week in Set A doesn't dominate. Predictions are multiplied back by `scale`.

**What to edit:** the configuration cell in §6 (`MODELS`, `ENSEMBLES`, `POSTPROCESS`, `N_WINDOWS`, `ORIGIN_STRIDE`). Every model is a function `fit_x(ctx, **options) -> (forecast matrix, fitted model)`, so adding one means writing one function and adding one line to `MODELS`.

**Runtime:** about 15–20 minutes on 16 cores with the defaults (4 windows + the final fit). LightGBM and the MLP take most of it. Use `N_WINDOWS = 1` or larger `ORIGIN_STRIDE` values for quick experiments.

In [ ]:
%pip install -q numpy pandas matplotlib pyarrow scikit-learn lightgbm

In [ ]:
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Works whether the kernel starts in the notebook folder or the project root
BASE = Path('.') if Path('set-a').exists() else Path('data/hackathon-retail-sets')
FEATURES_PATH = BASE / 'processed' / 'features_wide.parquet'
SUBMISSION_DIR = BASE / 'submissions'

H = 13
SID = 'series_id'
SEED = 0

# Plot style: same as the EDA notebooks
BLUE, GREY, INK = '#2a78d6', '#b5b4ae', '#52514e'
plt.rcParams.update({
    'figure.figsize': (11, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GREY, 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})

## 1. Load the feature frame and the raw test-shop history

- `frame`: one row per series × origin week with features and `target_h1 … target_h13` (preprocessing §8).
- `Y`: the raw test-shop matrix (articles × weeks). Scores are always computed on it, never on the feature frame.

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f'{FEATURES_PATH} not found: run preprocessing.ipynb first (its §11 saves it).')

frame = pd.read_parquet(FEATURES_PATH)
print(f'feature frame: {len(frame):,} rows x {frame.shape[1]} cols | series per source:',
      frame.groupby('source', observed=True)[SID].nunique().to_dict())

ts = pd.read_parquet(BASE / 'test-shop' / 'history.parquet')
ts['week_start'] = pd.to_datetime(ts['week_start'])
Y = ts.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)
print(f'test shop: {Y.shape[0]:,} articles x {Y.shape[1]} weeks ({Y.columns[0].date()} -> {Y.columns[-1].date()})')

## 2. Helpers (same as preprocessing.ipynb §9–10)

Copied here so this notebook runs on its own.

- `split_frame` / `subsample_origins` / `to_long` build the train / validation / prediction rows for a cutoff.
- `mase_score` is the competition metric (from the EDA notebook): w1–w13, the live w1–w7 and the final w8–w13.

In [ ]:
META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'store_id', 'article_id', 'scale', 'is_outlier',
             'sample_weight', 'supplier_id', 'product_group', 'division', 'department', 'department_group',
             'store_format', 'category', 'subcategory', 'segment']
LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'sample_weight']


def rolling_cutoffs(frame, n_windows=4, step=H, horizon=H, source='test_shop'):
    last = frame.loc[frame['source'] == source, 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def split_frame(frame, cutoff=None, horizon=H, valid_source='test_shop'):
    is_v = frame['source'] == valid_source
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    target_end = frame['week_start'] + pd.Timedelta(weeks=horizon)
    train = frame[~is_v | (target_end <= cutoff)]
    valid = frame[is_v & (frame['week_start'] == cutoff)]
    return train.reset_index(drop=True), valid.reset_index(drop=True)


def subsample_origins(frame, every=None, min_history_weeks=1):
    keep = pd.Series(True, index=frame.index)
    if every:
        k = frame['source'].astype(str).map(every).fillna(1).astype(int)
        from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
        keep &= (from_end % k == 0)
    if 'age_weeks' in frame:
        keep &= frame['age_weeks'] >= min_history_weeks - 1
    return frame[keep].reset_index(drop=True)


def to_long(frame, horizon=H, drop_missing_target=True, id_cols=LONG_ID_COLS):
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base = [c for c in id_cols if c in frame.columns] + feats
    parts = []
    for h in range(1, horizon + 1):
        p = frame[base].copy()
        p['h'] = np.int8(h)
        if f'seasonal_h{h}' in frame:
            p['seasonal_lag52'] = frame[f'seasonal_h{h}'].to_numpy()
        if f'target_h{h}' in frame:
            p['target'] = frame[f'target_h{h}'].to_numpy()
        parts.append(p)
    long = pd.concat(parts, ignore_index=True)
    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target and 'target' in long:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long


def forecast_matrix(rows, pred, index=Y.index, horizon=H):
    """Long rows (article_id, h) + predictions -> articles x horizon array; missing articles -> 0."""
    F = pd.DataFrame({'article_id': rows['article_id'].astype(str).to_numpy(), 'h': rows['h'].to_numpy(),
                      'pred': np.asarray(pred, dtype=float)}).pivot(index='article_id', columns='h', values='pred')
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


def per_article_mase(cutoff, F, Y=Y, horizon=H):
    """MASE per article (NaN for articles with no movement before the cutoff, which can't be scored)."""
    hist = Y.loc[:, :cutoff]
    act = Y.loc[:, Y.columns > cutoff].iloc[:, :horizon].to_numpy()
    den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
    err = np.abs(act - np.clip(F, 0, None))
    with np.errstate(divide='ignore', invalid='ignore'):
        return {name: np.where(den > 0, err[:, cols].mean(axis=1) / den, np.nan)
                for name, cols in [('all', slice(0, horizon)), ('w1-w7', slice(0, 7)), ('w8-w13', slice(7, horizon))]}


def mase_score(cutoff, F):
    """Competition score (1 - MASE) x 100, higher is better."""
    return {k: round((1 - np.nanmean(v)) * 100, 2) for k, v in per_article_mase(cutoff, F).items()}


def make_submission(F, index=Y.index):
    F = np.clip(np.asarray(F, dtype=float), 0, None)
    sub = pd.DataFrame(F, index=index, columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == 2904 and sub.article_id.is_unique and not sub.isna().any().any()
    return sub

## 3. Backtest context

`make_context` builds everything a model needs for one cutoff, once, so all models see exactly the same data:

| key | content |
|---|---|
| `cutoff` | last known week (`None` for the final fit = the last week of history) |
| `train` | long training rows: every origin of the extra sets, and test-shop origins whose 13 target weeks end by the cutoff |
| `valid` | long rows of the test shop at the cutoff (all 2,904 articles × 13 horizons) |
| `features` | model input columns |
| `Y_hist` | raw test-shop matrix up to the cutoff, for the local models |
| `weeks_since_sale`, `demand_class` | per article at the cutoff, for post-processing and the error breakdown |

In [ ]:
CAT_FEATURES = ['demand_class']


def make_context(frame, cutoff=None, origin_stride=None):
    train_w, valid_w = split_frame(frame, cutoff)
    cutoff = valid_w['week_start'].max()
    train = to_long(subsample_origins(train_w, every=origin_stride))
    valid = to_long(valid_w, drop_missing_target=False)
    features = [c for c in train.columns if c not in LONG_ID_COLS and c != 'target']
    at_cut = valid_w.assign(article_id=valid_w['article_id'].astype(str)).set_index('article_id').reindex(Y.index)
    return {
        'cutoff': cutoff, 'train': train, 'valid': valid, 'features': features,
        'Y_hist': Y.loc[:, :cutoff],
        'weeks_since_sale': at_cut['weeks_since_sale'].to_numpy(float),
        'demand_class': at_cut['demand_class'].astype(str).to_numpy(),
    }

## 4. Models

Each `fit_*` function trains on `ctx['train']` (or on `ctx['Y_hist']` for local models) and returns `(F, model)`: `F` is an articles × 13 array of forecasts in raw units, in the order of `Y.index`.

**Notes on the choices**
- **LightGBM**: `objective='l1'` because MASE is an absolute error. `demand_class` is a native categorical. Rows are weighted with `sample_weight` (test shop ×3, set in preprocessing).
- **Linear L1**: `SGDRegressor` with the epsilon-insensitive loss and `epsilon=0`, which is exactly the absolute loss. It needs clean inputs, so features go through `signed_log → median imputation (+ missing flags) → standard scaling`, and `demand_class` / `h` are one-hot encoded. It trains on a random sample of rows.
- **MLP**: scikit-learn's MLP only has squared loss. Training on `log1p(target)` turns that into something closer to a median for skewed demand. The log model still predicts small positive values for series that are mostly zero, and the median of such a week is 0, so predictions below `zero_below` (in scaled units) are set to 0.
- **TSB** (Teunter–Syntetos–Babai) smooths two things per article: the probability of a sale `p` (updated every week, so it decays when an article stops selling) and the size of a sale `z` (updated on sale weeks). Classic TSB forecasts the mean `p·z`. Under absolute error the best forecast is the *median*: `z` if `p > 0.5`, otherwise 0. That is `mode='median'`, which scored about 20 points better in testing.

In [ ]:
# ---------- reference ----------
def fit_median(ctx, weeks=13):
    f = ctx['Y_hist'].iloc[:, -weeks:].median(axis=1).to_numpy()
    return np.repeat(f[:, None], H, axis=1), None


# ---------- LightGBM ----------
LGB_PARAMS = dict(objective='l1', learning_rate=0.1, num_leaves=63, min_data_in_leaf=1000, max_bin=63,
                  feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=1.0,
                  force_col_wise=True, verbose=-1, seed=SEED)


def fit_lightgbm(ctx, params=LGB_PARAMS, rounds=200, use_weights=True):
    tr, va, feats = ctx['train'], ctx['valid'], ctx['features']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'] if use_weights else None,
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)
    pred = model.predict(va[feats]) * va['scale'].to_numpy()
    return forecast_matrix(va, pred), model


# ---------- shared input pipeline for the linear model and the MLP ----------
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_tabular_preprocessor(features, categorical=(*CAT_FEATURES, 'h')):
    cat = [c for c in categorical if c in features]
    num = [c for c in features if c not in cat]
    return ColumnTransformer([
        ('num', make_pipeline(FunctionTransformer(signed_log),
                              SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])


def _sample(df, n, seed=SEED):
    return df.sample(n=n, random_state=seed) if n and len(df) > n else df


# ---------- linear L1 ----------
def fit_linear_l1(ctx, sample=1_000_000, alpha=1e-5, epochs=10, target_clip_q=0.999, use_weights=True):
    tr = _sample(ctx['train'], sample)
    feats = ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, alpha=alpha, average=True,
                                       max_iter=epochs, tol=None, random_state=SEED))
    y = tr['target'].clip(upper=tr['target'].quantile(target_clip_q))   # scaled targets can explode for tiny scales
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], y, sgdregressor__sample_weight=tr['sample_weight'].to_numpy() if use_weights else None)
    va = ctx['valid']
    pred = model.predict(va[feats]).clip(0) * va['scale'].to_numpy()
    return forecast_matrix(va, pred), model


# ---------- MLP ----------
def fit_mlp(ctx, sample=500_000, hidden=(128, 64), max_iter=30, alpha=1e-4, zero_below=0.5):
    tr = _sample(ctx['train'], sample)
    feats = ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          MLPRegressor(hidden_layer_sizes=hidden, batch_size=2048, learning_rate_init=1e-3,
                                       alpha=alpha, max_iter=max_iter, early_stopping=True, n_iter_no_change=4,
                                       random_state=SEED))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], np.log1p(tr['target'].clip(lower=0)))
    va = ctx['valid']
    pred = np.expm1(model.predict(va[feats])).clip(0)
    pred = np.where(pred < zero_below, 0, pred) * va['scale'].to_numpy()
    return forecast_matrix(va, pred), model


# ---------- TSB ----------
def tsb_state(Y_hist, alpha=0.1, beta=0.05):
    """Run TSB over the history, vectorised across articles. Returns (p, z) at the last week."""
    X = Y_hist.to_numpy(float)
    p, z = np.zeros(len(X)), np.full(len(X), np.nan)
    for t in range(X.shape[1]):
        d = X[:, t]
        sale = d > 0
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = p + beta * (sale - p)
    return p, np.nan_to_num(z)


def fit_tsb(ctx, alpha=0.1, beta=0.05, mode='median', threshold=0.5):
    p, z = tsb_state(ctx['Y_hist'], alpha, beta)
    f = p * z if mode == 'mean' else np.where(p > threshold, z, 0.0)
    return np.repeat(f[:, None], H, axis=1), {'p': p, 'z': z}

## 5. Post-processing

Applied to every model's forecast after `clip ≥ 0`. Both raw and post-processed scores are reported, so you can see what each step does.

- `dead_to_zero`: forecast 0 for articles with no sale in the last `weeks` weeks (DATASETS §8.1; 1,073 articles at the end of history).
- `snap_small`: forecasts below `below` units → 0. Over-forecasting slow sellers is the biggest MASE risk (§2), and the median of a mostly-zero week is 0.

In [ ]:
def dead_to_zero(F, ctx, weeks=8):
    dead = ~(ctx['weeks_since_sale'] < weeks)          # NaN (never sold) counts as dead
    return np.where(dead[:, None], 0.0, F)


def snap_small(F, ctx, below=0.5):
    return np.where(F < below, 0.0, F)


def postprocess(F, ctx, steps):
    F = np.clip(F, 0, None)
    for step in steps:
        func, kw = (step, {}) if callable(step) else step
        F = func(F, ctx, **kw)
    return F

## 6. Configure ← edit here

- `MODELS`: name → `(fit function, options)`. Comment out a line to skip a model.
- `ENSEMBLES`: name → `(combine function, [model names])`. Built from the members' raw forecasts and post-processed like a model.
- `POSTPROCESS`: steps from §5, in order.
- `N_WINDOWS`: number of 13-week backtest windows (DATASETS §8.2 recommends several: scores move ~25 points between windows).
- `ORIGIN_STRIDE`: keep every k-th origin per source in the training rows. Smaller = more data, slower.

In [ ]:
MODELS = {
    'median_13w': (fit_median, {'weeks': 13}),
    'lightgbm':   (fit_lightgbm, {'rounds': 200}),
    'linear_l1':  (fit_linear_l1, {'sample': 1_000_000}),
    'mlp':        (fit_mlp, {'sample': 500_000, 'hidden': (128, 64)}),
    'tsb':        (fit_tsb, {'alpha': 0.1, 'beta': 0.05, 'mode': 'median'}),
}
REFERENCE = 'median_13w'

ENSEMBLES = {
    'ens_mean(lgb+linear)': (np.mean, ['lightgbm', 'linear_l1']),
    'ens_median(4 models)': (np.median, ['lightgbm', 'linear_l1', 'mlp', 'tsb']),
}

POSTPROCESS = [
    (dead_to_zero, {'weeks': 8}),
    (snap_small, {'below': 0.5}),
]

N_WINDOWS = 4
ORIGIN_STRIDE = {'test_shop': 2, 'A': 4, 'C': 4, 'D': 2}

## 7. Rolling backtest

For each cutoff: build the context, fit every model, score raw and post-processed forecasts. All forecasts are kept in `forecasts[(model, cutoff)]` for the ensembles and the analysis below. The fitted models of the most recent window are kept in `fitted`.

In [ ]:
def combine(forecasts, members, how, cutoff):
    return how(np.stack([forecasts[(m, cutoff)] for m in members]), axis=0)


def backtest(frame, models, ensembles, cutoffs, post_steps, origin_stride):
    rows, forecasts, fitted, contexts = [], {}, {}, {}
    for i, cutoff in enumerate(cutoffs):
        t0 = time.perf_counter()
        ctx = make_context(frame, cutoff, origin_stride)
        contexts[cutoff] = {k: v for k, v in ctx.items() if k not in ('train', 'valid')}
        print(f'\n=== cutoff {cutoff.date()} | train {len(ctx["train"]):,} rows | context {time.perf_counter() - t0:.0f}s')
        for name, (fit, kw) in models.items():
            t0 = time.perf_counter()
            F, model = fit(ctx, **kw)
            secs = time.perf_counter() - t0
            forecasts[(name, cutoff)] = F
            if i == 0:
                fitted[name] = model
            for variant, G in [('raw', np.clip(F, 0, None)), ('post', postprocess(F, ctx, post_steps))]:
                rows.append({'model': name, 'cutoff': cutoff.date(), 'variant': variant, 'fit_s': round(secs, 1),
                             **mase_score(cutoff, G)})
            print(f'  {name:<22s} raw {rows[-2]["all"]:7.2f} | post {rows[-1]["all"]:7.2f}   ({secs:.0f}s)')
        for name, (how, members) in ensembles.items():
            if all((m, cutoff) in forecasts for m in members):
                F = combine(forecasts, members, how, cutoff)
                forecasts[(name, cutoff)] = F
                for variant, G in [('raw', np.clip(F, 0, None)), ('post', postprocess(F, ctx, post_steps))]:
                    rows.append({'model': name, 'cutoff': cutoff.date(), 'variant': variant, 'fit_s': 0.0,
                                 **mase_score(cutoff, G)})
                print(f'  {name:<22s} raw {rows[-2]["all"]:7.2f} | post {rows[-1]["all"]:7.2f}')
        del ctx
    return pd.DataFrame(rows), forecasts, fitted, contexts


cutoffs = rolling_cutoffs(frame, n_windows=N_WINDOWS)
results, forecasts, fitted, contexts = backtest(frame, MODELS, ENSEMBLES, cutoffs, POSTPROCESS, ORIGIN_STRIDE)

### Results

Score = (1 − MASE) × 100, higher is better. For reference, the site's AutoARIMA scores −33.7 and the naive forecast −64.5. **`w8-w13` is what the final ranking uses** (DATASETS §2).

In [ ]:
summary = (results.groupby(['model', 'variant'])[['all', 'w1-w7', 'w8-w13', 'fit_s']].mean().round(2)
           .unstack('variant'))
summary.columns = [f'{m} ({v})' for m, v in summary.columns]
summary = summary.drop(columns=['fit_s (post)']).rename(columns={'fit_s (raw)': 'fit seconds'})
summary = summary.sort_values('all (post)', ascending=False)
print(f'Mean over {len(cutoffs)} windows:')
display(summary)

print('Per window (post-processed, all w1-w13):')
display(results[results.variant == 'post'].pivot(index='model', columns='cutoff', values='all')
        .loc[summary.index])

In [ ]:
s = summary['all (post)'].sort_values()
fig, ax = plt.subplots(figsize=(9, 0.45 * len(s) + 1.2))
colors = [GREY if m == REFERENCE else BLUE for m in s.index]
ax.barh(s.index, s.values, color=colors, height=0.6)
for y, v in enumerate(s.values):
    ax.text(v - 0.15 if v < 0 else v + 0.15, y, f'{v:.1f}', va='center', ha='right' if v < 0 else 'left',
            color=INK, fontsize=9)
ax.axvline(0, color=INK, linewidth=0.8)
ax.set_xlim(min(s.min() * 1.15, -1), max(s.max() * 1.15, 1))
ax.grid(axis='y', visible=False)
ax.set_xlabel('score = (1 − MASE) × 100, higher is better')
ax.set_title(f'Backtest score, mean of {len(cutoffs)} windows (post-processed; grey = reference)')
plt.tight_layout(); plt.show()

## 8. Where the error comes from

Mean MASE per demand class (at the cutoff) on the most recent window, post-processed. Lower is better here. DATASETS §8.6 suggests keeping whichever model wins per class. The `articles` column shows how many articles each class has, since every article weighs the same in the score.

In [ ]:
c0 = cutoffs[0]
ctx0 = contexts[c0]
models_shown = list(summary.index)
by_class = {}
for m in models_shown:
    G = postprocess(forecasts[(m, c0)], ctx0, POSTPROCESS)
    by_class[m] = pd.Series(per_article_mase(c0, G)['all']).groupby(ctx0['demand_class']).mean()
cls = pd.DataFrame(by_class).T.round(3)
cls.loc['articles'] = pd.Series(ctx0['demand_class']).value_counts()
display(cls[[c for c in ['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales'] if c in cls]])

### TSB parameter sensitivity

TSB is cheap (milliseconds per fit), so its parameters can be scanned over every window. These windows are the same ones used for scoring above, so choose robust settings rather than the single best cell.

In [ ]:
grid = []
for a in [0.05, 0.1, 0.2, 0.3]:
    for b in [0.02, 0.05, 0.1]:
        for thr in [0.4, 0.5, 0.6]:
            sc = [mase_score(c, postprocess(fit_tsb({'Y_hist': Y.loc[:, :c]}, a, b, 'median', thr)[0],
                                            contexts[c], POSTPROCESS))['all'] for c in cutoffs]
            grid.append({'alpha': a, 'beta': b, 'threshold': thr, 'mean score': round(np.mean(sc), 2),
                         'worst window': round(min(sc), 2)})
display(pd.DataFrame(grid).sort_values('mean score', ascending=False).head(10))

## 9. LightGBM feature importance

Gain importance from the model of the most recent window: how much each feature reduced the L1 loss in total.

In [ ]:
if 'lightgbm' in fitted:
    m = fitted['lightgbm']
    imp = pd.Series(m.feature_importance('gain'), index=m.feature_name()).sort_values()
    imp = (imp / imp.sum() * 100).tail(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('% of total gain')
    ax.set_title('LightGBM: top 20 features by gain (most recent window)')
    plt.tight_layout(); plt.show()

## 10. Final fit and submission

Refit on **all** history (cutoff = the last week, 2023-04-17), forecast 2023-04-24 → 2023-07-17 and write the CSV (`article_id, w1 … w13`).

`FINAL_MODEL` defaults to the best mean backtest score on w8–w13 (what the final ranking uses). Set it by hand to override. It can be a model or an ensemble.

In [ ]:
FINAL_MODEL = summary['w8-w13 (post)'].idxmax()
SAVE = True

members = ENSEMBLES[FINAL_MODEL][1] if FINAL_MODEL in ENSEMBLES else [FINAL_MODEL]
print(f'Final model: {FINAL_MODEL} (backtest mean: all {summary.loc[FINAL_MODEL, "all (post)"]}, '
      f'w8-w13 {summary.loc[FINAL_MODEL, "w8-w13 (post)"]})')

ctx_final = make_context(frame, cutoff=None, origin_stride=ORIGIN_STRIDE)
print(f'Training rows: {len(ctx_final["train"]):,} | forecast origin: {ctx_final["cutoff"].date()}')
final_fc = {}
for m in members:
    t0 = time.perf_counter()
    fit, kw = MODELS[m]
    final_fc[m] = fit(ctx_final, **kw)[0]
    print(f'  fitted {m} ({time.perf_counter() - t0:.0f}s)')

F_final = ENSEMBLES[FINAL_MODEL][0](np.stack([final_fc[m] for m in members]), axis=0) \
    if FINAL_MODEL in ENSEMBLES else final_fc[FINAL_MODEL]
F_final = postprocess(F_final, ctx_final, POSTPROCESS)
sub = make_submission(F_final)

print(f'Articles forecast 0 for all 13 weeks: {(F_final.sum(axis=1) == 0).sum():,} of {len(sub):,}')
print(f'Mean forecast per article-week: {F_final.mean():.2f} (last 13 weeks of history: {Y.iloc[:, -13:].to_numpy().mean():.2f})')
display(sub.head())

if SAVE:
    SUBMISSION_DIR.mkdir(exist_ok=True)
    path = SUBMISSION_DIR / f'submission_{re.sub(r"[^A-Za-z0-9]+", "_", FINAL_MODEL).strip("_")}.csv'
    sub.to_csv(path, index=False)
    print('Saved', path)